# Crossref Insight no Google Colab

Notebook para consultar a API Crossref e gerar:
- resumo do trabalho
- autores mais frequentes
- termos mais frequentes nos títulos
- gráficos de distribuição


In [ ]:
!pip -q install requests pandas matplotlib

In [ ]:
from collections import Counter
import re

import matplotlib.pyplot as plt
import pandas as pd
import requests

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
# Configuração da consulta
QUERY = "deep learning"
ROWS = 50
OFFSET = 0
MAILTO = "seu-email@exemplo.com"  # recomendado

SELECT_FIELDS = [
    "DOI", "title", "issued", "type", "publisher",
    "container-title", "author", "references-count",
    "is-referenced-by-count", "subject"
]

FILTERS = {
    "type": "journal-article",
    "has-references": "true"
}


In [ ]:
WORKS_URL = "https://api.crossref.org/works"
STOPWORDS = {
    "a","an","and","as","at","by","com","da","das","de","do","dos",
    "e","em","et","for","from","in","is","na","no","nos","of","on",
    "or","para","por","the","to","um","uma","with"
}

def filters_to_param(filters: dict) -> str:
    return ",".join(f"{k}:{v}" for k, v in filters.items())

def build_headers(mailto: str | None) -> dict:
    if mailto and mailto.strip():
        return {"User-Agent": f"CrossrefInsightColab/1.0 (mailto:{mailto.strip()})"}
    return {"User-Agent": "CrossrefInsightColab/1.0"}

def get_works(query: str, rows: int, offset: int, select_fields: list[str], filters: dict, mailto: str | None):
    params = {
        "query": query,
        "rows": rows,
        "offset": offset,
    }
    if select_fields:
        params["select"] = ",".join(select_fields)
    if filters:
        params["filter"] = filters_to_param(filters)
    if mailto and mailto.strip():
        params["mailto"] = mailto.strip()

    r = requests.get(WORKS_URL, params=params, headers=build_headers(mailto), timeout=30)
    js = r.json()
    return r.status_code, js, params

def get_year(item: dict):
    val = item.get("issued", {}).get("date-parts", [[None]])[0][0]
    return val if isinstance(val, int) else None

def get_authors(item: dict):
    out = []
    for a in item.get("author", []) or []:
        name = " ".join(filter(None, [a.get("given"), a.get("family")])).strip()
        if name:
            out.append(name)
    return out

def get_title(item: dict):
    titles = item.get("title", []) or []
    return titles[0] if titles else ""

def tokenize_title(title: str):
    tokens = re.findall(r"[a-zA-ZÀ-ÖØ-öø-ÿ0-9]+", title.lower())
    return [t for t in tokens if len(t) >= 3 and t not in STOPWORDS and not t.isdigit()]

def normalize_item(item: dict):
    return {
        "title": "; ".join(item.get("title", []) or []),
        "doi": item.get("DOI"),
        "authors": "; ".join(get_authors(item)),
        "container_title": "; ".join(item.get("container-title", []) or []),
        "publisher": item.get("publisher"),
        "year": get_year(item),
        "type": item.get("type"),
        "references_count": item.get("references-count"),
        "cited_by": item.get("is-referenced-by-count"),
    }


In [ ]:
status, js, sent_params = get_works(
    query=QUERY,
    rows=ROWS,
    offset=OFFSET,
    select_fields=SELECT_FIELDS,
    filters=FILTERS,
    mailto=MAILTO,
)

print("Status HTTP:", status)
if status != 200:
    print(js)

message = js.get("message", {})
items = message.get("items", [])
print("Items retornados:", len(items))
print("Total resultados na API:", message.get("total-results"))


In [ ]:
# Resumo do trabalho
years = [y for y in (get_year(i) for i in items) if y is not None]
authors_all = [a for i in items for a in get_authors(i)]
authors_counter = Counter(authors_all)
terms_counter = Counter(t for i in items for t in tokenize_title(get_title(i)))
type_counter = Counter(i.get("type") for i in items if i.get("type"))
publisher_counter = Counter(i.get("publisher") for i in items if i.get("publisher"))
journal_counter = Counter((i.get("container-title") or [""])[0] for i in items if (i.get("container-title") or [""])[0])

summary = {
    "query": QUERY,
    "rows_requested": ROWS,
    "filters": FILTERS,
    "status": status,
    "returned_items": len(items),
    "total_results": message.get("total-results"),
    "unique_dois": len({i.get("DOI") for i in items if i.get("DOI")}),
    "unique_authors": len(set(authors_all)),
    "year_min": min(years) if years else None,
    "year_max": max(years) if years else None,
}

summary


In [ ]:
top_authors_df = pd.DataFrame(authors_counter.most_common(15), columns=["author", "frequency"])
top_terms_df = pd.DataFrame(terms_counter.most_common(20), columns=["term", "frequency"])

display(top_authors_df)
display(top_terms_df)


In [ ]:
# Gráficos
year_df = pd.DataFrame(sorted(Counter(years).items()), columns=["year", "count"])
type_df = pd.DataFrame(type_counter.most_common(10), columns=["type", "count"])
publisher_df = pd.DataFrame(publisher_counter.most_common(10), columns=["publisher", "count"])
journal_df = pd.DataFrame(journal_counter.most_common(10), columns=["journal", "count"])

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

if not year_df.empty:
    axes[0,0].plot(year_df["year"], year_df["count"], marker="o")
axes[0,0].set_title("Publicações por ano")
axes[0,0].set_xlabel("Ano")
axes[0,0].set_ylabel("Quantidade")

if not type_df.empty:
    axes[0,1].bar(type_df["type"], type_df["count"])
axes[0,1].set_title("Tipos de documento")
axes[0,1].tick_params(axis="x", rotation=60)

if not top_authors_df.empty:
    axes[0,2].bar(top_authors_df["author"].head(10), top_authors_df["frequency"].head(10))
axes[0,2].set_title("Top autores")
axes[0,2].tick_params(axis="x", rotation=75)

if not top_terms_df.empty:
    axes[1,0].bar(top_terms_df["term"].head(12), top_terms_df["frequency"].head(12))
axes[1,0].set_title("Termos frequentes nos títulos")
axes[1,0].tick_params(axis="x", rotation=65)

if not publisher_df.empty:
    axes[1,1].bar(publisher_df["publisher"], publisher_df["count"])
axes[1,1].set_title("Top editoras")
axes[1,1].tick_params(axis="x", rotation=75)

if not journal_df.empty:
    axes[1,2].bar(journal_df["journal"], journal_df["count"])
axes[1,2].set_title("Top periódicos")
axes[1,2].tick_params(axis="x", rotation=75)

plt.tight_layout()
plt.show()


In [ ]:
# Tabela final normalizada
df = pd.DataFrame([normalize_item(i) for i in items])
df.head(20)


In [ ]:
# Exportação opcional
import json

df.to_csv("crossref_results.csv", index=False)
top_authors_df.to_csv("top_authors.csv", index=False)
top_terms_df.to_csv("top_terms.csv", index=False)

with open("summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Arquivos gerados: crossref_results.csv, top_authors.csv, top_terms.csv, summary.json")
